# Thí nghiệm cặp sinh đôi · Notebook 2/3 · 12 job Mono/Multi, 5 epoch, seed 17 · khoảng 3,5 giờ

**Chạy một lần, ra một file zip:** `twin_nb2.zip`. Gửi file này lại là xong.

Ba notebook **độc lập**, chạy song song trên 3 tài khoản (mỗi tài khoản một notebook). **Chỉ chạy sau khi bản bổ sung lần 3 đã đăng trên OSF.**

**Thí nghiệm:** giống hệt lần chạy chia ngẫu nhiên trước (bản bổ sung lần 1): cùng 18.694 hàm v1, cùng mã huấn luyện, seed 17, 5 epoch, cùng danh sách job. Thay đổi duy nhất: 930 cặp sinh đôi (2 hàm cùng ngôn ngữ, cùng cụm CVE, ngược nhãn) trước đây bị tách sang hai fold nay được xếp vào cùng một fold, bằng cách chuyển một hàm sang fold của hàm kia. Mọi hàm khác giữ nguyên fold. Phân công fold được nhúng sẵn trong notebook và kiểm tra bằng mã băm.

**Chuẩn bị:**
1. Add Input → dataset `mvd-grouped-cv-inputs` (dataset cũ, không cần dataset mới).
2. Settings: Accelerator = **GPU T4 x2**, **Internet = On**.
3. **Save Version → Save & Run All (Commit)**. Kết quả ở tab Output: `twin_nb2.zip`.

## 1. Môi trường

In [ ]:
import time
NB_START = time.time()
!nvidia-smi
!pip install -q transformers scikit-learn pandas numpy
import torch, transformers, sklearn, pandas
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Chưa bật GPU: Settings → Accelerator → GPU T4'

## 2. Mã nguồn, dữ liệu thô

In [ ]:
import glob, os, shutil, zipfile
from pathlib import Path
REPO = Path('/kaggle/working/repo')
if not (REPO / 'src/preprocessing/dataset_loader.py').exists():
    hits = glob.glob('/kaggle/input/**/src/preprocessing/dataset_loader.py', recursive=True)
    if hits:
        shutil.copytree(Path(hits[0]).parents[2], REPO)
    else:
        zips = glob.glob('/kaggle/input/**/multilingual-vulnerability-detection*.zip', recursive=True)
        assert zips, 'Không thấy repo trong /kaggle/input'
        tmp = Path('/kaggle/working/_repo_tmp'); tmp.mkdir(exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z: z.extractall(tmp)
        inner = [p for p in tmp.iterdir() if p.is_dir()]
        shutil.move(str(inner[0] if len(inner) == 1 else tmp), str(REPO))
raw_dir = REPO / 'data/raw'; raw_dir.mkdir(parents=True, exist_ok=True)
NEED = ['train_w_line.csv', 'val_w_line.csv', 'test_w_line.csv']
if not all((raw_dir / n).exists() for n in NEED):
    # Kaggle có thể đã tự giải nén data_raw_for_colab.zip thành thư mục: tìm 3 file CSV ở bất kỳ đâu
    found = {}
    for n in NEED:
        cands = [Path(p) for p in glob.glob(f'/kaggle/input/**/{n}', recursive=True)]
        cands = [p for p in cands if p.stat().st_size > 1_000_000]
        if cands: found[n] = cands[0]
    if len(found) == len(NEED):
        for n, src in found.items(): shutil.copy2(src, raw_dir / n)
    else:  # trường hợp còn nguyên file zip
        zp = list(REPO.rglob('data_raw_for_colab.zip')) + [Path(p) for p in glob.glob('/kaggle/input/**/data_raw_for_colab.zip', recursive=True)]
        assert zp, 'Không thấy 3 file *_w_line.csv cũng không thấy data_raw_for_colab.zip trong /kaggle/input'
        with zipfile.ZipFile(zp[0]) as z: z.extractall(raw_dir)
import pandas as pd
rows = sum(len(pd.read_csv(raw_dir / n, usecols=[0])) for n in NEED)
print('Dữ liệu thô:', {n: (raw_dir / n).stat().st_size // 1_000_000 for n in NEED}, 'MB | tổng dòng:', rows)
assert rows == 20165, f'Số dòng {rows} != 20165: sai file dữ liệu thô'
FOLDS_V1 = glob.glob('/kaggle/input/**/groups_and_folds_v1.csv', recursive=True)[0]
print('File fold v1 (chỉ dùng để lấy danh sách hàm và cụm CVE):', FOLDS_V1)
os.chdir(REPO)

## 3. Tạo file fold "giữ cặp sinh đôi" và kiểm tra

In [ ]:
import base64, hashlib, numpy as np, pandas as pd
ASSIGN_B64 = """
...
"""
v1 = pd.read_csv(FOLDS_V1).sort_values('unit_id').reset_index(drop=True)
assert len(v1) == 18694, 'Sai file groups_and_folds_v1.csv'
code = np.frombuffer(base64.b64decode(''.join(ASSIGN_B64.split())), dtype=np.uint8)
assert len(code) == len(v1)
out = v1[['unit_id', 'language', 'target']].copy()
blk = v1.groupby(['language', 'group']).agg(n=('unit_id', 'size'), nt=('target', 'nunique')).reset_index()
blk = blk[(blk.n == 2) & (blk.nt == 2)][['language', 'group']]
is_twin = v1.merge(blk, on=['language', 'group'], how='left', indicator=True)['_merge'].eq('both').values
out['group'] = np.where(is_twin, 'twin:' + v1.language + ':' + v1.group.astype(str), v1.unit_id)
out['is_giant'] = False
out['fold'] = (code // 32).astype(int)
FLAGS = [f'inner_val_fold{k}' for k in range(5)]
for k in range(5):
    out[FLAGS[k]] = ((code >> k) & 1).astype(bool)
lines = (out.unit_id + ',' + out.fold.astype(str) + ',' + out[FLAGS].astype(int).astype(str).agg(''.join, axis=1) + ',' + out.group.astype(str))
h = hashlib.sha256('\n'.join(lines).encode()).hexdigest()
tw = out[is_twin]
print('Số cặp sinh đôi:', tw.group.nunique(), '| cặp nằm trọn một fold:', int(tw.groupby('group').fold.nunique().eq(1).sum()))
print('Số hàm mỗi fold:', out.fold.value_counts().sort_index().to_dict(), '| mã băm:', h)
assert tw.group.nunique() == 1157 and tw.groupby('group').fold.nunique().eq(1).all()
assert h == 'b44e1857ff72d677eb991f623a8696fd9bae298bef4e327aa74fd71532174468', 'Phân công fold không khớp bản đã đăng ký'
FOLDS = '/kaggle/working/twin_folds_v1.csv'
out.to_csv(FOLDS, index=False)

## 4. Ghi mã vào repo (nguyên văn mã đã dùng ở các lần trước)

In [ ]:
%%writefile /kaggle/working/repo/src/training/grouped_cv_splits.py
"""
Chọn dữ liệu cho Grouped 5-fold CV (v1). Chỉ dùng pandas, không cần GPU.

Đầu vào:
  clean  : DataFrame từ MVDDatasetLoader().clean_exact_hash() (18.694 đơn vị) có cột unit_id, func, language, target.
  folds  : groups_and_folds_v1.csv (cột unit_id, group, is_giant, fold, inner_val_fold0..4).

Quy tắc (mọi phép chia đều theo CỤM `group` = thành phần liên thông của CVE, xuyên ngôn ngữ):
  mono(L, k)  : test = đơn vị của L ở fold k; val = đơn vị của L ở cụm có cờ inner_val_fold{k};
                train = đơn vị của L ở fold != k và không phải val.
  multi(k)    : như mono nhưng gộp cả 7 ngôn ngữ.
  lolo(T)     : test = TOÀN BỘ đơn vị của ngôn ngữ T (không cần chia). Nguồn = 6 ngôn ngữ còn lại,
                bỏ mọi đơn vị nguồn cùng cụm với một đơn vị của T (khử nhiễm theo CVE);
                val = đơn vị nguồn có cờ inner_val_fold0 (các cụm ngoài fold 0, ~9%); train = phần còn lại.
Mọi hàm đều kiểm tra: không cụm nào xuất hiện ở hai tập khác nhau.
"""
from __future__ import annotations

import pandas as pd

LANGUAGES = ["C", "C#", "C++", "Go", "Java", "JavaScript", "Python"]
N_FOLDS = 5


def attach_folds(clean: pd.DataFrame, folds: pd.DataFrame) -> pd.DataFrame:
    need_clean = {"unit_id", "func", "language", "target"}
    need_folds = {"unit_id", "group", "is_giant", "fold"} | {f"inner_val_fold{k}" for k in range(N_FOLDS)}
    if need_clean - set(clean.columns):
        raise ValueError(f"clean thiếu cột: {sorted(need_clean - set(clean.columns))}")
    if need_folds - set(folds.columns):
        raise ValueError(f"folds thiếu cột: {sorted(need_folds - set(folds.columns))}")
    if clean["unit_id"].duplicated().any() or folds["unit_id"].duplicated().any():
        raise ValueError("unit_id bị trùng")
    cols = ["unit_id", "group", "is_giant", "fold"] + [f"inner_val_fold{k}" for k in range(N_FOLDS)]
    df = clean.merge(folds[cols], on="unit_id", how="inner", validate="one_to_one")
    if len(df) != len(clean) or len(df) != len(folds):
        raise ValueError(f"unit_id không khớp: clean={len(clean)}, folds={len(folds)}, merged={len(df)}")
    # nhãn và ngôn ngữ trong file fold phải khớp với dữ liệu sạch
    chk = folds.set_index("unit_id")
    for col in ("language", "target"):
        if col in chk.columns:
            a = df.set_index("unit_id")[col]
            b = chk[col].reindex(a.index)
            if not (a.astype(str) == b.astype(str)).all():
                raise ValueError(f"cột {col} lệch giữa clean và folds")
    for k in range(N_FOLDS):
        df[f"inner_val_fold{k}"] = df[f"inner_val_fold{k}"].astype(bool)
    df["is_giant"] = df["is_giant"].astype(bool)
    return df.reset_index(drop=True)


def _assert_disjoint_groups(**parts: pd.DataFrame) -> None:
    names = list(parts)
    groups = {n: set(parts[n]["group"]) for n in names}
    units = {n: set(parts[n]["unit_id"]) for n in names}
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            inter = groups[a] & groups[b]
            if inter:
                raise AssertionError(f"RÒ RỈ: {len(inter)} cụm xuất hiện ở cả {a} và {b}")
            if units[a] & units[b]:
                raise AssertionError(f"RÒ RỈ: unit_id xuất hiện ở cả {a} và {b}")


def split_mono(df: pd.DataFrame, language: str, k: int):
    d = df[df["language"] == language]
    val_flag = d[f"inner_val_fold{k}"]
    test = d[d["fold"] == k]
    val = d[(d["fold"] != k) & val_flag]
    train = d[(d["fold"] != k) & ~val_flag]
    _assert_disjoint_groups(train=train, val=val, test=test)
    return train.copy(), val.copy(), test.copy()


def split_multi(df: pd.DataFrame, k: int):
    val_flag = df[f"inner_val_fold{k}"]
    test = df[df["fold"] == k]
    val = df[(df["fold"] != k) & val_flag]
    train = df[(df["fold"] != k) & ~val_flag]
    _assert_disjoint_groups(train=train, val=val, test=test)
    return train.copy(), val.copy(), test.copy()


def split_lolo(df: pd.DataFrame, target_language: str):
    tgt = df[df["language"] == target_language]
    src = df[df["language"] != target_language]
    contaminated = src["group"].isin(set(tgt["group"]))
    n_removed = int(contaminated.sum())
    src = src[~contaminated]
    val_flag = src["inner_val_fold0"]
    val = src[val_flag]
    train = src[~val_flag]
    _assert_disjoint_groups(train=train, val=val, test=tgt)
    assert (train["language"] != target_language).all() and (val["language"] != target_language).all()
    return train.copy(), val.copy(), tgt.copy(), n_removed


def build_jobs(df: pd.DataFrame, modes=("mono", "multi", "lolo"), languages=None, folds=None):
    """Danh sách job (mode, key, fold) theo thứ tự chạy: multi/mono theo fold rồi ngôn ngữ, lolo theo ngôn ngữ."""
    languages = list(languages or LANGUAGES)
    folds = list(range(N_FOLDS)) if folds is None else list(folds)
    jobs = []
    if "mono" in modes:
        for k in folds:
            for lang in languages:
                jobs.append(("mono", lang, k))
    if "multi" in modes:
        for k in folds:
            jobs.append(("multi", "ALL", k))
    if "lolo" in modes:
        for lang in languages:
            jobs.append(("lolo", lang, -1))
    return jobs


def select_jobs(all_jobs, specs):
    """Chọn job theo danh sách chuỗi 'mode:key:fold' (vd 'mono:C#:0', 'multi:ALL:2', 'lolo:Java:-1'), giữ thứ tự đã cho."""
    index = {(m, key, k) for m, key, k in all_jobs}
    out = []
    for s in specs:
        parts = s.split(":")
        if len(parts) != 3:
            raise ValueError(f"job sai định dạng (cần mode:key:fold): {s!r}")
        job = (parts[0], parts[1], int(parts[2]))
        if job not in index:
            raise ValueError(f"job không tồn tại: {s!r}")
        if job in out:
            raise ValueError(f"job bị lặp: {s!r}")
        out.append(job)
    return out


def get_split(df: pd.DataFrame, mode: str, key: str, k: int):
    if mode == "mono":
        return (*split_mono(df, key, k), 0)
    if mode == "multi":
        return (*split_multi(df, k), 0)
    if mode == "lolo":
        return split_lolo(df, key)
    raise ValueError(mode)


def plan_table(df: pd.DataFrame, epochs: int = 5, jobs=None) -> pd.DataFrame:
    rows = []
    for mode, key, k in (jobs or build_jobs(df)):
        tr, va, te, removed = get_split(df, mode, key, k)
        rows.append(dict(mode=mode, key=key, fold=k, n_train=len(tr), n_val=len(va), n_test=len(te),
                         pos_train=float(tr["target"].mean()), removed_for_decontamination=removed,
                         sample_passes=len(tr) * epochs))
    return pd.DataFrame(rows)


In [ ]:
%%writefile /kaggle/working/repo/src/training/run_grouped_cv.py
"""
Huấn luyện Grouped 5-fold CV (mono / multi) và LOLO trên tập sạch 18.694 đơn vị, dùng lại engine của repo.
Đặt file này ở src/training/run_grouped_cv.py (cùng thư mục với grouped_cv_splits.py).

Giữ nguyên công thức đã đóng băng của repo (UniXcoder, AdamW lr 2e-5, batch 8, 5 epoch, max_length 768, fp32,
warmup 10%, clip 1.0). Thay đổi DUY NHẤT so với repo cũ:
  - checkpoint chọn theo AUROC validation (macro theo ngôn ngữ), không dùng F1 tại ngưỡng 0,5;
  - chia dữ liệu theo cụm CVE (grouped_cv_splits.py), validation nội bộ cố định theo file fold.
Tập test của mỗi job chỉ được dự đoán MỘT lần, bằng checkpoint tốt nhất trên validation.
Không lưu checkpoint ra đĩa; chỉ lưu dự đoán xác suất + nhật ký.
"""
from __future__ import annotations

import argparse
import gc
import json
import sys
import time
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from torch import nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

from src.preprocessing.dataset_loader import MVDDatasetLoader
from src.preprocessing.tokenization import UniXcoderInputPipeline
from src.training import grouped_cv_splits as S
from src.training.modeling import UniXcoderBinaryClassifier
from src.training.training_engine import (
    EncodedPilotDataset,
    PilotBatchCollator,
    build_train_loader,
    build_validation_loader,
    optimizer_step_count,
    preencode_frame,
    set_global_seed,
    train_one_epoch,
    warmup_step_count,
)

CONFIG_PATH = REPO_ROOT / "configs" / "rq1_mono_training_v1.json"
CHECKPOINT_METRIC = "val_auroc_macro_by_language"


def env_info() -> dict:
    import transformers
    return dict(gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                torch=torch.__version__, transformers=transformers.__version__)


def clean_key(key: str) -> str:
    return key.replace("#", "sharp").replace("+", "plus").lower()


def job_dir(out: Path, mode: str, key: str, k: int, seed: int) -> Path:
    return out / mode / clean_key(key) / (f"fold{k}" if k >= 0 else "all") / f"seed_{seed}"


def job_done(d: Path) -> bool:
    return (d / "predictions.csv").is_file() and (d / "log.json").is_file()


class RecordCache:
    """Mã hoá từng đơn vị đúng một lần cho cả phiên chạy."""

    def __init__(self, pipeline: UniXcoderInputPipeline) -> None:
        self.pipeline = pipeline
        self.records: dict[str, object] = {}

    def dataset(self, frame: pd.DataFrame) -> EncodedPilotDataset:
        missing = frame[~frame["unit_id"].isin(self.records)]
        if len(missing):
            ds = preencode_frame(missing[["unit_id", "func", "language", "target"]], self.pipeline)
            for i in range(len(ds)):
                rec = ds[i]
                self.records[rec.unit_id] = rec
        return EncodedPilotDataset([self.records[u] for u in frame["unit_id"]])


@torch.inference_mode()
def predict(model: nn.Module, loader, device: torch.device, criterion: nn.Module):
    model.eval()
    ids, langs, ys, ps = [], [], [], []
    loss_sum, n = 0.0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss_sum += float(criterion(logits, labels).item()) * int(labels.shape[0])
        n += int(labels.shape[0])
        ps.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
        ys.extend(labels.cpu().tolist())
        ids.extend(batch["unit_ids"])
        langs.extend(batch["languages"])
    return pd.DataFrame({"unit_id": ids, "language": langs, "true_label": ys, "prob_vulnerable": ps}), loss_sum / max(n, 1)


def macro_auroc(frame: pd.DataFrame) -> float:
    vals = []
    for _, g in frame.groupby("language"):
        if g["true_label"].nunique() == 2:
            vals.append(roc_auc_score(g["true_label"], g["prob_vulnerable"]))
    return float(np.mean(vals)) if vals else float("nan")


def run_job(mode, key, k, seed, df, cache, cfg, device, out: Path, split=None, epochs=None) -> dict:
    d = job_dir(out, mode, key, k, seed)
    tr, va, te, removed = split if split is not None else S.get_split(df, mode, key, k)
    ex = cfg["training_execution"]
    bsz = int(ex["physical_batch_size"])
    max_epochs = int(epochs) if epochs is not None else int(ex["max_epochs"])
    accum, clip = int(ex["gradient_accumulation_steps"]), float(ex["gradient_clipping_max_norm"])

    set_global_seed(seed)
    collator = PilotBatchCollator(cache.pipeline)
    train_loader = build_train_loader(cache.dataset(tr), collator, bsz, seed=seed)
    val_loader = build_validation_loader(cache.dataset(va), collator, bsz)
    test_loader = build_validation_loader(cache.dataset(te), collator, bsz)

    mcfg = cfg["model"]
    model = UniXcoderBinaryClassifier(
        model_name=mcfg["name"], revision=mcfg["revision"],
        expected_hidden_size=int(mcfg["classification_head"]["hidden_size"]),
        num_labels=int(mcfg["classification_head"]["num_labels"]),
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    o = cfg["optimizer"]
    optimizer = AdamW(model.parameters(), lr=float(o["learning_rate"]),
                      weight_decay=float(o["weight_decay"]), eps=float(o["adam_epsilon"]))
    total = optimizer_step_count(len(train_loader), accum, max_epochs)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_step_count(total, float(ex["warmup_ratio"])), num_training_steps=total)

    t0 = time.time()
    best_score, best_epoch, best_state, log = -float("inf"), -1, None, []
    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device, accum, clip)
        vpred, vloss = predict(model, val_loader, device, criterion)
        auroc = macro_auroc(vpred)
        score = auroc if not np.isnan(auroc) else -vloss  # chỉ xảy ra nếu validation chỉ có một lớp
        f1 = float(((vpred.prob_vulnerable >= 0.5) & (vpred.true_label == 1)).sum() * 2 /
                   max(((vpred.prob_vulnerable >= 0.5).sum() + (vpred.true_label == 1).sum()), 1))
        log.append(dict(epoch=epoch, train_loss=float(train_loss), val_loss=float(vloss),
                        val_auroc_macro=None if np.isnan(auroc) else float(auroc), val_f1_at_0p5=f1))
        print(f"[{mode}/{key}/fold{k}/seed{seed}] epoch {epoch}/{max_epochs} train_loss={train_loss:.4f} "
              f"val_loss={vloss:.4f} val_auroc={auroc:.4f}", flush=True)
        if score > best_score:  # nghiêm ngặt: hoà thì giữ epoch sớm hơn
            best_score, best_epoch = score, epoch
            best_state = {n: p.detach().to("cpu", copy=True) for n, p in model.state_dict().items()}

    model.load_state_dict(best_state)
    tpred, _ = predict(model, test_loader, device, criterion)  # test: một lần duy nhất
    meta = te.set_index("unit_id")[["group", "fold"]]
    tpred = tpred.join(meta, on="unit_id")
    tpred["mode"], tpred["seed"] = mode, seed
    d.mkdir(parents=True, exist_ok=True)
    tpred.to_csv(d / "predictions.csv", index=False)
    summary = dict(mode=mode, key=key, outer_fold=k, seed=seed, n_train=len(tr), n_val=len(va), n_test=len(te),
                   removed_for_decontamination=removed, selected_epoch=best_epoch,
                   checkpoint_metric=CHECKPOINT_METRIC, best_val_score=float(best_score),
                   seconds=round(time.time() - t0, 1), env=env_info(), epochs=log)
    (d / "log.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    del model, optimizer, scheduler, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary


def smoke(df, cache, cfg, device, out: Path, report_dir: Path) -> None:
    """Chạy thử toàn bộ đường đi (train -> chọn checkpoint -> dự đoán -> ghi file) trên mẫu rất nhỏ.
    Dùng các đơn vị VALIDATION của mono/Python/fold 0 làm 'test giả', KHÔNG dùng test thật, kết quả bị bỏ."""
    tr, va, _, _ = S.get_split(df, "mono", "Python", 0)
    split = (tr.head(48), va.head(24), va.iloc[24:48], 0)
    res = run_job("smoke", "Python", 0, 0, df, cache, cfg, device, out, split=split, epochs=2)
    pred = pd.read_csv(job_dir(out, "smoke", "Python", 0, 0) / "predictions.csv")
    assert len(pred) == 24 and pred["prob_vulnerable"].between(0, 1).all()
    info = {k: res[k] for k in ("n_train", "n_val", "n_test", "selected_epoch", "seconds")}
    report_dir.mkdir(parents=True, exist_ok=True)
    (report_dir / "smoke_report.json").write_text(json.dumps(dict(status="SMOKE OK", **info, env=env_info(),
                                                                  epochs=res["epochs"]), indent=2), encoding="utf-8")
    print("SMOKE OK:", info)


def pilot(df, cache, cfg, device, steps: int, report_dir: Path) -> None:
    """Đo tốc độ trên dữ liệu trộn 7 ngôn ngữ (train của multi, fold 0); KHÔNG chạm tập test."""
    mode, key, k = "multi", "ALL", 0
    tr, va, _, _ = S.get_split(df, mode, key, k)
    ex = cfg["training_execution"]
    bsz = int(ex["physical_batch_size"])
    collator = PilotBatchCollator(cache.pipeline)
    loader = build_train_loader(cache.dataset(tr), collator, bsz, seed=0)
    mcfg = cfg["model"]
    model = UniXcoderBinaryClassifier(
        model_name=mcfg["name"], revision=mcfg["revision"],
        expected_hidden_size=int(mcfg["classification_head"]["hidden_size"]),
        num_labels=int(mcfg["classification_head"]["num_labels"])).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=float(cfg["optimizer"]["learning_rate"]))
    model.train()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    seen, t0, warm = 0, None, 3
    for i, batch in enumerate(loader):
        if i == warm:
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0, seen = time.time(), 0
        logits = model(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
        loss = criterion(logits, batch["labels"].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        seen += int(batch["labels"].shape[0])
        if i + 1 >= warm + steps:
            break
    if device.type == "cuda":
        torch.cuda.synchronize()
    rate = seen / (time.time() - t0)
    vsub = va.head(64)
    vpred, vloss = predict(model, build_validation_loader(cache.dataset(vsub), collator, bsz), device, criterion)
    plan = S.plan_table(df, epochs=int(ex["max_epochs"]))
    passes = float(plan["sample_passes"].sum())
    hours = passes / rate / 3600 * 1.1  # +10% cho validation/test
    report = dict(samples_per_second=round(rate, 2), steps_measured=steps, batch_size=bsz,
                  peak_gpu_mem_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else None,
                  sample_passes_per_seed=passes, est_hours_per_seed=round(hours, 2),
                  est_hours_per_seed_by_mode={m: round(float(plan.loc[plan["mode"] == m, "sample_passes"].sum()) / rate / 3600 * 1.1, 2)
                                              for m in ("mono", "multi", "lolo")},
                  est_hours_per_job={f"{r['mode']}|{r['key']}|{r['fold']}": round(r["sample_passes"] / rate / 3600 * 1.1, 3)
                                     for _, r in plan.iterrows()},
                  val_probe_loss=round(vloss, 4), env=env_info())
    report_dir.mkdir(parents=True, exist_ok=True)
    (report_dir / "pilot_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("=" * 70)
    print(f"Tốc độ huấn luyện đo được: {rate:.1f} mẫu/giây  (peak GPU mem: "
          f"{torch.cuda.max_memory_allocated() / 1e9:.1f} GB)" if device.type == "cuda" else f"Tốc độ: {rate:.1f} mẫu/giây (CPU)")
    print(f"Đường dự đoán validation chạy được: {len(vpred)} mẫu, loss={vloss:.3f}")
    print(f"Tổng số lượt mẫu-epoch mỗi seed (mono+multi+lolo): {passes:,.0f}")
    print(f"Ước tính thời gian MỖI SEED: {hours:.1f} giờ GPU")
    print(f"Ước tính 3 seed: {3 * hours:.1f} giờ  (hạn mức Kaggle ~30 giờ/tuần, mỗi phiên tối đa 9 giờ)")
    print("=" * 70)


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--folds-csv", type=Path, required=True)
    ap.add_argument("--out", type=Path, default=Path("/kaggle/working/grouped_cv_results"))
    ap.add_argument("--modes", nargs="*", default=["mono", "multi", "lolo"])
    ap.add_argument("--seeds", nargs="*", type=int, default=[17])
    ap.add_argument("--languages", nargs="*", default=None)
    ap.add_argument("--folds", nargs="*", type=int, default=None)
    ap.add_argument("--jobs", nargs="*", default=None,
                    help="chỉ chạy các job này, dạng mode:key:fold (vd mono:C#:0 multi:ALL:1 lolo:Java:-1); thay cho --modes/--languages/--folds")
    ap.add_argument("--max-hours", type=float, default=8.3, help="dừng sạch trước giới hạn 9 giờ của phiên Kaggle")
    ap.add_argument("--pilot", action="store_true")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--pilot-steps", type=int, default=60)
    ap.add_argument("--report-dir", type=Path, default=Path("/kaggle/working/pilot_outputs"))
    ap.add_argument("--plan-only", action="store_true")
    args = ap.parse_args()

    cfg = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    clean = MVDDatasetLoader().clean_exact_hash(copy=True)
    df = S.attach_folds(clean, pd.read_csv(args.folds_csv))
    if args.jobs:
        jobs = S.select_jobs(S.build_jobs(df), args.jobs)
    else:
        jobs = S.build_jobs(df, modes=args.modes, languages=args.languages, folds=args.folds)
    plan = S.plan_table(df, epochs=int(cfg["training_execution"]["max_epochs"]), jobs=jobs)
    print(plan.groupby("mode")[["n_train", "n_val", "n_test", "sample_passes"]].sum())
    print(f"Số job mỗi seed: {len(jobs)}")
    if args.plan_only:
        return 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Thiết bị:", device)
    cache = RecordCache(UniXcoderInputPipeline())
    if args.smoke:
        import tempfile
        smoke(df, cache, cfg, device, Path(tempfile.mkdtemp()), args.report_dir)
        return 0
    if args.pilot:
        pilot(df, cache, cfg, device, args.pilot_steps, args.report_dir)
        return 0

    start, done_passes, done_secs = time.time(), 0.0, 0.0
    for seed in args.seeds:
        for (mode, key, k), (_, row) in zip(jobs, plan.iterrows()):
            d = job_dir(args.out, mode, key, k, seed)
            if job_done(d):
                print(f"[BỎ QUA] đã xong: {d}")
                continue
            elapsed = (time.time() - start) / 3600
            est = (row["sample_passes"] / (done_passes / done_secs) / 3600 * 1.1) if done_secs > 0 else 0.0
            if elapsed + est > args.max_hours:
                print(f"[DỪNG] đã chạy {elapsed:.2f} giờ, job kế tiếp cần ~{est:.2f} giờ, vượt {args.max_hours} giờ. "
                      "Lưu kết quả rồi chạy lại cell này ở phiên sau (tự bỏ qua job đã xong).")
                return 0
            res = run_job(mode, key, k, seed, df, cache, cfg, device, args.out)
            done_passes += float(row["sample_passes"])
            done_secs += res["seconds"]
    print("HOÀN TẤT các job được yêu cầu.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


## 5. (Chỉ dùng khi phiên bị ngắt giữa chừng) nạp kết quả dở dang
Bình thường cell này không làm gì. Nếu phiên bị ngắt: Add Input là output của bản commit bị ngắt rồi chạy lại; job đã xong được bỏ qua.

In [ ]:
OUT = Path('/kaggle/working/twin_cv_results'); OUT.mkdir(exist_ok=True)
n = 0
for p in [Path(x) for x in glob.glob('/kaggle/input/**/twin_cv_results', recursive=True)]:
    for f in p.rglob('*'):
        if f.is_file():
            dst = OUT / f.relative_to(p)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst); n += 1
for z in glob.glob('/kaggle/input/**/twin_nb2.zip', recursive=True):   # trường hợp input là file zip
    with zipfile.ZipFile(z) as zz:
        for name in zz.namelist():
            if name.startswith('twin_cv_results/') and not name.endswith('/'):
                dst = Path('/kaggle/working') / name
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True); dst.write_bytes(zz.read(name)); n += 1
runs_file = OUT / 'runs.txt'
RUN = (len(runs_file.read_text().split()) if runs_file.exists() else 0) + 1
print('Đã nạp', n, 'file dở dang' if n else 'file (bình thường, chạy mới)')

## 6. Chạy thử nhanh (khoảng 3–5 phút)
Huấn luyện 2 epoch trên 48 mẫu validation, kết quả bị bỏ. Phải in `SMOKE OK`.

In [ ]:
import subprocess
r = subprocess.run(['python', 'src/training/run_grouped_cv.py', '--folds-csv', FOLDS, '--smoke'],
                   capture_output=True, text=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), cwd=str(REPO))
print(r.stdout[-3000:]); print(r.stderr[-3000:])
assert 'SMOKE OK' in r.stdout, 'Chạy thử thất bại: xem lỗi ở trên, chụp gửi lại'

## 7. Huấn luyện trên 2 GPU
Mỗi GPU chạy một danh sách job (khoảng 3–3,5 giờ). Cứ 5 phút in một dòng tiến độ.

In [ ]:
import subprocess, os, time
SESSION_HOURS = 8.3      # giới hạn an toàn cho phiên 9 giờ; tính từ lúc notebook bắt đầu
SEEDS = "17"
LANES = [["multi:ALL:2", "mono:Python:1", "mono:Python:3", "mono:C:3", "mono:C++:0", "mono:C++:3"], ["multi:ALL:0", "mono:Java:0", "mono:Java:4", "mono:C:1", "mono:Go:3", "mono:C#:3"]]
n_gpu = int(subprocess.check_output('nvidia-smi -L | wc -l', shell=True).decode().strip())
assert n_gpu == 2, f'Cần Accelerator = GPU T4 x2 (đang thấy {n_gpu} GPU). Settings → Accelerator.'
def remaining_hours():
    return max(0.1, SESSION_HOURS - (time.time() - NB_START) / 3600)
procs = []
for i, jobs in enumerate(LANES):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), TOKENIZERS_PARALLELISM='false')
    cmd = ['python', 'src/training/run_grouped_cv.py', '--folds-csv', FOLDS, '--out', str(OUT), '--jobs', *jobs,
           '--seeds', *SEEDS.split(), '--max-hours', f'{remaining_hours():.2f}']
    log = open(f'/kaggle/working/lane{i}_run{RUN}.log', 'w')
    procs.append(subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, cwd=str(REPO)))
    print(f'lane {i}: GPU {i}, {len(jobs)} job'); time.sleep(45)
def tail(i):
    try:
        t = open(f'/kaggle/working/lane{i}_run{RUN}.log', errors='ignore').read().split('\n')
        t = [x.split('\r')[-1] for x in t if x.strip()]
        return t[-1][:180] if t else ''
    except Exception as e:
        return str(e)
try:
    while any(p.poll() is None for p in procs):
        time.sleep(300)
        print(f'[{(time.time() - NB_START) / 3600:.2f} giờ]')
        for i in range(len(procs)): print(f'  lane{i}:', tail(i))
        print('  GPU:', subprocess.run('nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip().replace('\n', ' | '))
finally:
    for p in procs:
        if p.poll() is None: p.terminate()
print('Mã thoát:', [p.returncode for p in procs])
for i in range(len(procs)):
    print(f'--- cuối lane{i}_run{RUN}.log ---'); print('\n'.join([x for x in open(f'/kaggle/working/lane{i}_run{RUN}.log', errors='ignore').read().split('\n') if x.strip()][-5:]))

## 8. Kiểm tra và đóng gói (zip gồm cả kết quả các lần trước nếu có)

In [ ]:
import json
with open(OUT / 'runs.txt', 'a') as fh: fh.write(f'run{RUN}\n')
for i in range(2):
    src = Path(f'/kaggle/working/lane{i}_run{RUN}.log')
    if src.exists(): shutil.copy2(src, OUT / f'nb2_{src.name}')
shutil.copy2(FOLDS, OUT / 'twin_folds_v1.csv')
ck = lambda k: k.replace('#', 'sharp').replace('+', 'plus').lower()
done, left = [], []
for j in [j for l in LANES for j in l]:
    m, k, f = j.split(':')
    d = OUT / m / ck(k) / f'fold{f}' / 'seed_17'
    ok = (d / 'predictions.csv').exists() and (d / 'log.json').exists()
    (done if ok else left).append(j)
    if ok:
        lg = json.load(open(d / 'log.json')); print(f"{m:5s} {k:10s} fold={f} epoch={lg['selected_epoch']} {lg['seconds'] / 60:.0f} phút")
print(f'Đã xong {len(done)}/{len(done) + len(left)} job của notebook 2')
if left:
    print('CÒN LẠI:', left)
    print('→ Hiếm khi xảy ra. Gửi file zip này kèm ảnh chụp, mình sẽ hướng dẫn chạy nốt.')
else:
    print('NOTEBOOK 2 ĐÃ XONG TOÀN BỘ. Gửi file twin_nb2.zip.')
name = 'twin_nb2'
shutil.make_archive(f'/kaggle/working/{name}', 'zip', '/kaggle/working', 'twin_cv_results')
from IPython.display import FileLink, display
display(FileLink(f'/kaggle/working/{name}.zip'))